# Tugas 2 | Crawling pta.trunojoyo.ac.id

## Versi Pertama (ptaa())

- Loop halaman terbatas
Hanya mengambil 3 halaman pertama (for i in range(1, 4)).

- Proses serial (satu per satu)
Setiap link jurnal dibuka bergantian, tidak ada paralelisasi.

- Struktur hasil → langsung dikumpulkan dalam dictionary, lalu diubah jadi DataFrame.

- Output CSV → disimpan dengan nama pta.csv.

Berikut Code Crawling nya:

**Import Library**

In [6]:
!pip install requests
!pip install beautifulsoup4
import requests
from bs4 import BeautifulSoup
import pandas as pd

**Fungsi crawling sederhana (sequential)**

- Ambil data dari 3 halaman pertama saja
- Proses detail jurnal satu per satu (lambat tapi simpel)

In [ ]:
def ptaa():
    # siapkan struktur data untuk menyimpan hasil
    data = {"penulis": [], "judul": [], "pembimbing_pertama": [], "pembimbing_kedua": [], "abstrak": []}

    # loop dari halaman 1 sampai 3
    for i in range(1, 4):
        url = "https://pta.trunojoyo.ac.id/c_search/byfac/4/{}".format(i)
        r = requests.get(url)
        request = r.content
        soup = BeautifulSoup(request, "html.parser")

        # ambil semua item jurnal di halaman
        jurnals = soup.select('li[data-cat="#luxury"]')

        # proses setiap jurnal di halaman
        for jurnal in jurnals:
            # buka halaman detail jurnal
            response = requests.get(jurnal.select_one('a.gray.button')['href'])
            soup1 = BeautifulSoup(response.content, "html.parser")

            isi = soup1.select_one('div#content_journal')

            # ambil data penting dari halaman detail
            judul = isi.select_one('a.title').text
            penulis = isi.select_one('span:contains("Penulis")').text.split(' : ')[1]
            pembimbing_pertama = isi.select_one('span:contains("Dosen Pembimbing I")').text.split(' : ')[1]
            pembimbing_kedua = isi.select_one('span:contains("Dosen Pembimbing II")').text.split(' :')[1]

            # ambil abstrak (fallback kalau kosong)
            abstrak = isi.select_one('p[align="justify"]').text
            if abstrak == '':
                abstrak = ' '.join(isi.find('p').findNext('p').stripped_strings).capitalize()

            # simpan ke dictionary
            data["penulis"].append(penulis)
            data["judul"].append(judul)
            data["pembimbing_pertama"].append(pembimbing_pertama)
            data["pembimbing_kedua"].append(pembimbing_kedua)
            data["abstrak"].append(abstrak)

    # konversi hasil ke DataFrame
    df = pd.DataFrame(data)

    # simpan ke file CSV
    df.to_csv("pta.csv", index=False)
    return df

In [10]:
ptaa()

/usr/local/lib/python3.12/dist-packages/soupsieve/css_parser.py:876: FutureWarning: The pseudo class ':contains' is deprecated, ':-soup-contains' should be used moving forward.
  warnings.warn(  # noqa: B028


,penulis,judul,pembimbing_pertama,pembimbing_kedua,abstrak
0,A.Ubaidillah S.Kom,PERANCANGAN DAN IMPLEMENTASI SISTEM DATABASE \...,Budi Setyono M.T,Hermawan S.T,Sistem informasi akademik (SIAKAD) merupaka...
1,"M. Basith Ardianto,",APLIKASI KONTROL DAN MONITORING JARINGAN KOMPU...,"Drs. Budi Soesilo, MT","Koko Joni, ST",Berjalannya koneksi jaringan komputer dengan l...
2,"Akhmad Suyandi, S.Kom",RANCANG BANGUN APLIKASI PROXY SERVER UNTUK\r\n...,"Drs. Budi Soesilo, M.T","Hermawan, ST, MT",Web server adalah sebuah perangkat lunak serve...
3,Heri Supriyanto,SISTEM PENDUKUNG KEPUTUSAN OPTIMASI PENJADWALA...,"Mulaab, S.Si., M.Kom","Firli Irhamni, ST., M.Kom",Penjadwalan kuliah di Perguruan Tinggi me...
4,Septian Rahman Hakim,SISTEM AUGMENTED REALITY ANIMASI BENDA BERGERA...,"Arik Kurniawati, S.Kom., M.T.","Haryanto, S.T., M.T.",Seiring perkembangan teknologi yang ada diduni...
5,Adi Chandra Laksono,Gerak Pekerja Pada Game Real Time Strategy Men...,"Kurniawan Eka P, S.Kom., Msc","Arik Kurniawati, S.Kom., M.T.",Gerak pekerja ada pada game yang memiliki genr...
6,NURRACHMAT,RANCANG BANGUN GAME PERAWATAN SAPI KARAPAN MEN...,"Arik Kurniawati, S.Kom., M.T.","Kurniawan Eka Permana, S.Kom., MSc.","Perkembangan game yang semakin pesat, memberik..."
7,Muhammad Choirur Rozi,EKSTRAKSI FITUR BERBASIS TWO DIMENSIONAL LINEA...,"Dr. Arif Muntasa, S.Si.,M.T","Fitri Damayanti, S.Kom.,M.kom",Sistem pengenalan wajah adalah suatu sistem un...
8,M Khoiril Anwar,IMPLEMENTASI ALGORITMA PRIM DAN DEPTH FIRST ...,"Cucun Very Angkoso, S.T., M.T.","Arik Kurniawati S. Kom., M.T.",Teknologi mobile game beroperating system open...
9,Elky Lilik Misdayani,SISTEM INFORMASI PERUSAHAAN OTOBUS SUMBER KENC...,"Fitri Damayanti,S.Kom,M.Kom",,Sistem informasi pengolahan data yang dibutuhk...


## Versi Kedua (ptaa_multithreaded())

* Loop otomatis semua halaman

    Jalan terus sampai halaman terakhir (tidak ada batasan manual 1–3).
    
    Dicek dengan if not jurnals: break.

- Multithreading

    Menggunakan ThreadPoolExecutor agar beberapa detail jurnal bisa di-fetch paralel → jauh lebih cepat.

- Ada delay antar halaman (time.sleep(2)) supaya tidak dianggap spam oleh server.

- Output CSV → disimpan dengan nama pta_mulitT_full.csv.

- Lebih robust, ada penanganan error di crawl_detail.



**Import Library**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

**Fungsi bantu untuk ambil detail satu jurnal**

Input: link jurnal

Output: dict (penulis, judul, pembimbing, abstrak)

In [ ]:
def crawl_detail(link):
    try:
        response = requests.get(link, timeout=10)
        soup1 = BeautifulSoup(response.content, "html.parser")
        isi = soup1.select_one('div#content_journal')

        # parsing data penting
        judul = isi.select_one('a.title').text.strip()
        penulis = isi.select_one('span:contains("Penulis")').text.split(' : ')[1].strip()
        pembimbing_pertama = isi.select_one('span:contains("Dosen Pembimbing I")').text.split(' : ')[1].strip()
        pembimbing_kedua = isi.select_one('span:contains("Dosen Pembimbing II")').text.split(' :')[1].strip()

        # ambil abstrak, fallback kalau kosong
        abstrak = isi.select_one('p[align="justify"]').text.strip()
        if abstrak == '':
            abstrak = ' '.join(isi.find('p').findNext('p').stripped_strings).capitalize()

        return {
            "penulis": penulis,
            "judul": judul,
            "pembimbing_pertama": pembimbing_pertama,
            "pembimbing_kedua": pembimbing_kedua,
            "abstrak": abstrak
        }
    except Exception as e:
        print(f"❌ Gagal crawling {link}: {e}")
        return None

**Fungsi utama crawling (multithreaded)**

- Ambil data semua halaman sampai habis
- Proses detail jurnal secara paralel dengan ThreadPoolExecutor

In [ ]:
def ptaa_multithreaded(max_workers=5):
    data = []
    page = 1

    while True:
        # ambil daftar jurnal dari halaman
        url = f"https://pta.trunojoyo.ac.id/c_search/byfac/4/{page}"
        r = requests.get(url)
        soup = BeautifulSoup(r.content, "html.parser")
        jurnals = soup.select('li[data-cat="#luxury"]')

        # jika tidak ada lagi jurnal, selesai
        if not jurnals:
            print(f"✅ Selesai! Halaman terakhir: {page-1}")
            break

        print(f"🔎 Crawling halaman {page}, jumlah jurnal: {len(jurnals)}")

        # kumpulkan semua link detail
        links = [jurnal.select_one('a.gray.button')['href'] for jurnal in jurnals]

        # jalankan request detail jurnal secara paralel
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(crawl_detail, link) for link in links]
            for future in as_completed(futures):
                result = future.result()
                if result:
                    data.append(result)

        # delay antar halaman untuk menghindari server overload
        time.sleep(2)
        page += 1

    # konversi hasil ke DataFrame
    df = pd.DataFrame(data)

    # simpan ke file CSV
    df.to_csv("pta_mulitT_full.csv", index=False)
    return df

In [15]:
ptaa_multithreaded()

🔎 Crawling halaman 1, jumlah jurnal: 5
🔎 Crawling halaman 2, jumlah jurnal: 5
🔎 Crawling halaman 3, jumlah jurnal: 5
🔎 Crawling halaman 4, jumlah jurnal: 5
🔎 Crawling halaman 5, jumlah jurnal: 5
🔎 Crawling halaman 6, jumlah jurnal: 5
🔎 Crawling halaman 7, jumlah jurnal: 5
🔎 Crawling halaman 8, jumlah jurnal: 5
🔎 Crawling halaman 9, jumlah jurnal: 5
🔎 Crawling halaman 10, jumlah jurnal: 5
🔎 Crawling halaman 11, jumlah jurnal: 5
🔎 Crawling halaman 12, jumlah jurnal: 5
🔎 Crawling halaman 13, jumlah jurnal: 5
🔎 Crawling halaman 14, jumlah jurnal: 5
🔎 Crawling halaman 15, jumlah jurnal: 5
🔎 Crawling halaman 16, jumlah jurnal: 5
🔎 Crawling halaman 17, jumlah jurnal: 5
🔎 Crawling halaman 18, jumlah jurnal: 5
❌ Gagal crawling https://pta.trunojoyo.ac.id/welcome/detail/070411100023: HTTPSConnectionPool(host='pta.trunojoyo.ac.id', port=443): Read timed out. (read timeout=10)
❌ Gagal crawling https://pta.trunojoyo.ac.id/welcome/detail/090451100037: HTTPSConnectionPool(host='pta.trunojoyo.ac.id', 

/tmp/ipython-input-3461295233.py:21: DeprecationWarning: Call to deprecated method findNext. (Replaced by find_next) -- Deprecated since version 4.0.0.
  abstrak = ' '.join(isi.find('p').findNext('p').stripped_strings).capitalize()


🔎 Crawling halaman 38, jumlah jurnal: 5
🔎 Crawling halaman 39, jumlah jurnal: 5
🔎 Crawling halaman 40, jumlah jurnal: 5
🔎 Crawling halaman 41, jumlah jurnal: 5
🔎 Crawling halaman 42, jumlah jurnal: 5
🔎 Crawling halaman 43, jumlah jurnal: 5
🔎 Crawling halaman 44, jumlah jurnal: 5
🔎 Crawling halaman 45, jumlah jurnal: 5
🔎 Crawling halaman 46, jumlah jurnal: 5
🔎 Crawling halaman 47, jumlah jurnal: 5
🔎 Crawling halaman 48, jumlah jurnal: 5
🔎 Crawling halaman 49, jumlah jurnal: 5
🔎 Crawling halaman 50, jumlah jurnal: 5
🔎 Crawling halaman 51, jumlah jurnal: 5
🔎 Crawling halaman 52, jumlah jurnal: 5
🔎 Crawling halaman 53, jumlah jurnal: 5
🔎 Crawling halaman 54, jumlah jurnal: 5
🔎 Crawling halaman 55, jumlah jurnal: 5
🔎 Crawling halaman 56, jumlah jurnal: 5
🔎 Crawling halaman 57, jumlah jurnal: 5
🔎 Crawling halaman 58, jumlah jurnal: 5
🔎 Crawling halaman 59, jumlah jurnal: 5
🔎 Crawling halaman 60, jumlah jurnal: 5
🔎 Crawling halaman 61, jumlah jurnal: 5
🔎 Crawling halaman 62, jumlah jurnal: 5


,penulis,judul,pembimbing_pertama,pembimbing_kedua,abstrak
0,A.Ubaidillah S.Kom,PERANCANGAN DAN IMPLEMENTASI SISTEM DATABASE \...,Budi Setyono M.T,Hermawan S.T,Sistem informasi akademik (SIAKAD) merupaka...
1,"M. Basith Ardianto,",APLIKASI KONTROL DAN MONITORING JARINGAN KOMPU...,"Drs. Budi Soesilo, MT","Koko Joni, ST",Berjalannya koneksi jaringan komputer dengan l...
2,"Akhmad Suyandi, S.Kom",RANCANG BANGUN APLIKASI PROXY SERVER UNTUK\r\n...,"Drs. Budi Soesilo, M.T","Hermawan, ST, MT",Web server adalah sebuah perangkat lunak serve...
3,Septian Rahman Hakim,SISTEM AUGMENTED REALITY ANIMASI BENDA BERGERA...,"Arik Kurniawati, S.Kom., M.T.","Haryanto, S.T., M.T.",Seiring perkembangan teknologi yang ada diduni...
4,Heri Supriyanto,SISTEM PENDUKUNG KEPUTUSAN OPTIMASI PENJADWALA...,"Mulaab, S.Si., M.Kom","Firli Irhamni, ST., M.Kom",Penjadwalan kuliah di Perguruan Tinggi me...
...,...,...,...,...,...
2281,Lilis Kurnia Ikamawati,PENENTUAN TINGKAT PRIORITAS PERBAIKAN MODE KEG...,"Dr. Kukuh Winarso, S.Si., M.T, IPM. Asean Eng","Issa Dyah Utami, S.T., M.T.",Penelitian ini berlokasikan di PT Semen Indone...
2282,MUNIF SAYID ANWAS PRABOWO,Evaluasi Tingkat Kelelahan Kerja Perawat Ruang...,"Dr. Weny Findiastuti, S.T.,M.T","Rullie Annisa, S.T.,M.T",Rumah sakit merupakan salah satu instansi di b...
2283,Friska Fatmawatiningrum,IDENTIFIKASI BINER ATRIBUT PEJALAN KAKI MENGGU...,"Dr. Indah Agustien Siradjuddin, S.Kom., M.Kom.","Prof. Dr. Arief Muntasa, S.Si., M.MT.",Identifikasi atribut pejalan kaki merupakan sa...
2284,Dian Wibowo,DETEKSI OBJEK MANUSIA BERBASIS ONE STAGE DETEC...,"Dr. Indah Agustien Siradjuddin, S.Kom., M.Kom.","Moch. Kautsar Sophan, S.Kom., M.MT.",Topik deteksi objek telah menarik perhatian ya...
